# Silver — web-shop orders (3 tables)

One Bronze table, one JSON parse, **three Silver tables** — because the document
contains three different grains:

| Table | Grain | SCD | Why |
| --- | --- | --- | --- |
| `silver.sales_orders` | one order | 2 | the source re-emits an order when lines are added, and we want that history |
| `silver.sales_order_lines` | one order × line | 1 | lines only ever get added; the latest version of the order wins |
| `silver.sales_order_clicks` | one order × clicked product | 1 | click-stream, its own grain |

The promotion is **not** its own table: `promotion_info.promo_item` always equals
the line's own product, so it describes the line rather than existing separately.
It becomes columns on the line, and Gold builds a small `dim_promotion` for labels.

In [ ]:
import sys
from datetime import date
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "common_utils").is_dir():
        sys.path.insert(0, str(candidate))
        break

from pyspark.sql import functions as F

from common_utils.logger import get_logger, log_info
from common_utils.observability import ensure_ops_schema, new_run_id, track
from common_utils.scd import business_columns, deduplicate, row_hash, scd1_merge, scd2_merge
from common_utils.settings import parse_run_date
from common_utils.transforms import add_derived, cast_columns, drop_columns, explode_array, normalise_nulls, parse_json_column, rename_columns, trim_columns
from common_utils.writers import cluster_by, create_namespace, qualified, set_table_properties

In [ ]:
dbutils.widgets.text("catalog", "retaildataplatform")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("run_date", date.today().isoformat())

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
run_date = parse_run_date(dbutils.widgets.get("run_date"))
run_id = new_run_id()
logger = get_logger("silver")

create_namespace(spark, catalog, silver_schema, comment="Silver: cleaned, typed and de-duplicated entities with change history")
ensure_ops_schema(spark, catalog)

# The shape of the nested arrays, written once and reused by every table below.
LINE_SCHEMA = (
    "array<struct<curr:string,id:string,name:string,price:string,qty:string,unit:string,"
    "promotion_info:struct<promo_disc:double,promo_id:string,promo_item:string,promo_qty:string>>>"
)
CLICK_SCHEMA = "array<array<string>>"
PROMO_SCHEMA = "array<struct<promo_disc:double,promo_id:string,promo_item:string,promo_qty:string>>"

## 1. Read Bronze and parse the JSON once
`_id` is the Cosmos document id. It increases with insertion time, so it is what
tells us which copy of a re-emitted order is the newest.

In [ ]:
bronze = spark.table(f"{catalog}.{bronze_schema}.sales_orders").filter(F.col("_load_date") == F.lit(run_date).cast("date"))
print("bronze rows for", run_date, ":", bronze.count())

base = normalise_nulls(bronze)
base = trim_columns(base, ["customer_name"])
base = rename_columns(base, {"_id": "source_document_id"})
base = cast_columns(base, {"order_number": "bigint", "customer_id": "bigint", "number_of_line_items": "int", "order_datetime": "bigint"})
base = add_derived(
    base,
    {
        "order_ts": "CAST(order_datetime AS TIMESTAMP)",
        "order_date": "CAST(CAST(order_datetime AS TIMESTAMP) AS DATE)",
    },
)
base = parse_json_column(base, "ordered_products", LINE_SCHEMA)
base = parse_json_column(base, "clicked_items", CLICK_SCHEMA)
base = parse_json_column(base, "promo_info", PROMO_SCHEMA)
base = drop_columns(base, ["order_datetime", "_ingested_at", "_source_file"])

# One document wins per order, and it wins for all three tables. Cosmos re-sends an order
# as a new document when it changes, and an older document can carry lines the new one no
# longer has: de-duplicating the header alone would leave those orphan lines behind and
# the line totals would no longer add up to the header total.
base = deduplicate(base, keys=["order_number"], order_by="source_document_id")

display(base.select("order_number", "customer_id", "order_ts", "number_of_line_items").limit(5))

## 2. `silver.sales_orders` — the header (SCD2)
Counts and totals are derived here, *before* the arrays are dropped, so the header
can be queried without touching the child tables.

`aggregate(array, start, (acc, x) -> ...)` is a for-loop written in SQL: it walks
the line items adding `price × qty`.

In [ ]:
header = add_derived(
    base,
    {
        "line_item_count": "size(ordered_products)",
        "has_promotion": "size(promo_info) > 0",
        "clicked_item_count": "size(clicked_items)",
        "click_count": "CAST(aggregate(clicked_items, 0L, (acc, x) -> acc + CAST(x[1] AS BIGINT)) AS BIGINT)",
        "order_gross_amount": (
            "CAST(aggregate(ordered_products, CAST(0 AS DOUBLE), (acc, p) -> "
            "acc + CAST(p.price AS DOUBLE) * CAST(p.qty AS DOUBLE)) AS DECIMAL(14,2))"
        ),
    },
)
header = drop_columns(header, ["ordered_products", "clicked_items", "promo_info"])

with track(spark, catalog, run_id, run_date, task="sales_orders_silver", layer="silver", entity="sales_orders") as stats:
    hashed = row_hash(header, business_columns(header, exclude=["source_document_id"]))
    latest = deduplicate(hashed, keys=["order_number"], order_by="source_document_id")
    prepared = latest.withColumn("_updated_at", F.current_timestamp())

    stats.rows_read = prepared.count()
    scd2_merge(spark, prepared, qualified(catalog, silver_schema, "sales_orders"), keys=["order_number"])
    stats.rows_written = stats.rows_read

    set_table_properties(spark, catalog, silver_schema, "sales_orders")
    cluster_by(spark, catalog, silver_schema, "sales_orders", ["order_number"])
    log_info(logger, "silver sales_orders merged", rows=stats.rows_read)

## 3. `silver.sales_order_lines` — one row per line (SCD1)
`explode_array(..., position_column="line_number")` turns one order with three
products into three rows numbered 1, 2, 3.

De-duplicating on `(order_number, line_number)` ordered by `source_document_id`
means a re-emitted order contributes only its newest set of lines.

Amounts are computed **here, once**, so Gold and every analyst use the same
arithmetic: `net = qty × price × (1 − discount)`.

In [ ]:
lines = explode_array(base, "ordered_products", alias="p", position_column="line_number")
lines = add_derived(
    lines,
    {
        "product_id": "p.id",
        "product_name": "p.name",
        "quantity": "try_cast(p.qty AS INT)",
        "unit_price": "try_cast(p.price AS DECIMAL(12,2))",
        "currency": "p.curr",
        "unit": "p.unit",
        "promo_id": "coalesce(p.promotion_info.promo_id, 'NONE')",
        "promo_discount_rate": "coalesce(p.promotion_info.promo_disc, 0)",
        "promo_quantity": "try_cast(p.promotion_info.promo_qty AS INT)",
        "gross_amount": "CAST(try_cast(p.qty AS INT) * try_cast(p.price AS DECIMAL(12,2)) AS DECIMAL(14,2))",
        "discount_amount": (
            "CAST(try_cast(p.qty AS INT) * try_cast(p.price AS DECIMAL(12,2)) "
            "* coalesce(p.promotion_info.promo_disc, 0) AS DECIMAL(14,2))"
        ),
        "net_amount": (
            "CAST(try_cast(p.qty AS INT) * try_cast(p.price AS DECIMAL(12,2)) "
            "* (1 - coalesce(p.promotion_info.promo_disc, 0)) AS DECIMAL(14,2))"
        ),
    },
)
lines = drop_columns(lines, ["p", "clicked_items", "promo_info", "customer_name", "number_of_line_items"])

with track(spark, catalog, run_id, run_date, task="sales_orders_silver", layer="silver", entity="sales_order_lines") as stats:
    hashed = row_hash(lines, business_columns(lines, exclude=["source_document_id"]))
    deduped = deduplicate(hashed, keys=["order_number", "line_number"], order_by="source_document_id")
    prepared = deduped.withColumn("_updated_at", F.current_timestamp())

    stats.rows_read = prepared.count()
    scd1_merge(spark, prepared, qualified(catalog, silver_schema, "sales_order_lines"), keys=["order_number", "line_number"])
    stats.rows_written = stats.rows_read

    set_table_properties(spark, catalog, silver_schema, "sales_order_lines")
    cluster_by(spark, catalog, silver_schema, "sales_order_lines", ["order_number"])
    log_info(logger, "silver sales_order_lines merged", rows=stats.rows_read)

## 4. `silver.sales_order_clicks` — the click-stream (SCD1)
Each element is a two-item array `[product_id, click_count]`, so the fields come
out by position rather than by name.

In [ ]:
clicks = explode_array(base, "clicked_items", alias="c", position_column="click_rank")
clicks = add_derived(clicks, {"product_id": "c[0]", "click_count": "try_cast(c[1] AS INT)"})
clicks = drop_columns(clicks, ["c", "ordered_products", "promo_info", "customer_name", "number_of_line_items"])

with track(spark, catalog, run_id, run_date, task="sales_orders_silver", layer="silver", entity="sales_order_clicks") as stats:
    hashed = row_hash(clicks, business_columns(clicks, exclude=["source_document_id"]))
    deduped = deduplicate(hashed, keys=["order_number", "product_id"], order_by="source_document_id")
    prepared = deduped.withColumn("_updated_at", F.current_timestamp())

    stats.rows_read = prepared.count()
    scd1_merge(spark, prepared, qualified(catalog, silver_schema, "sales_order_clicks"), keys=["order_number", "product_id"])
    stats.rows_written = stats.rows_read

    set_table_properties(spark, catalog, silver_schema, "sales_order_clicks")
    cluster_by(spark, catalog, silver_schema, "sales_order_clicks", ["order_number"])
    log_info(logger, "silver sales_order_clicks merged", rows=stats.rows_read)

## 5. Does it add up?
The sum of the line amounts must equal the header total we derived from the array.
If these ever differ, the flattening is wrong.

In [ ]:
display(
    spark.sql(
        f"""
        SELECT o.order_gross_amount_total, l.line_gross_amount_total,
               o.order_gross_amount_total = l.line_gross_amount_total AS reconciles
        FROM (SELECT sum(order_gross_amount) AS order_gross_amount_total FROM {catalog}.{silver_schema}.sales_orders WHERE is_current) o
        CROSS JOIN (SELECT sum(gross_amount) AS line_gross_amount_total FROM {catalog}.{silver_schema}.sales_order_lines) l
        """
    )
)